# Unified Alpha Fusion Development

This notebook is generated from the development repository and imports the canonical implementation from `src/bigalpha2026/alpha_models`; it does not contain a handwritten second model. Model training uses expanding history from 2019, while 60 days is only the temporal sequence length. The upstream artifact must contain all 462 candidates through 2024 before the formal suite starts.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bigalpha2026.alpha_models import (
    ALL618,
    BAR156,
    DEFAULT_SUBMISSION_DATA_CONTRACT,
    All156TemporalConfig,
    All156TemporalNetwork,
    candidate_ids_from_manifest,
)

CANDIDATE_POOL = Path('/root/autodl-tmp/candidate462_completion_full_2019_2024/candidate462_store/features')
CANDIDATE_MANIFEST = Path('/root/autodl-tmp/candidate462_completion_full_2019_2024/candidate462_store/candidate462_manifest.json')
manifest = json.loads(CANDIDATE_MANIFEST.read_text(encoding="utf-8"))
candidate_ids = candidate_ids_from_manifest(CANDIDATE_MANIFEST)
date_range = manifest.get("date_range")
candidate_ready = (
    CANDIDATE_POOL.exists()
    and len(candidate_ids) == ALL618.candidate_feature_count
    and isinstance(date_range, list)
    and len(date_range) == 2
    and str(date_range[1]) >= "2024-12-31"
)
print({
    "candidate_count": len(candidate_ids),
    "expected_candidate_count": ALL618.candidate_feature_count,
    "candidate_ready": candidate_ready,
    "date_range": date_range,
    "candidate_pool": str(CANDIDATE_POOL),
})

## Contracts

In [ ]:
contract = DEFAULT_SUBMISSION_DATA_CONTRACT
print({
    "allowed_factor_sources": ["bar1m", "financial"],
    "training_start": str(contract.training_start),
    "training_end": str(contract.training_end),
    "temporal_lookback_days": contract.temporal_lookback_days,
    "bar_feature_count": BAR156.total_feature_count,
    "all_feature_count": ALL618.total_feature_count,
})

## Canonical model

In [ ]:
config = All156TemporalConfig(
    input_dim=156,
    candidate_dim=462,
    candidate_hidden_dim=256,
    model_dim=256,
    lookback=60,
    kernels=(3, 5, 15),
    transformer_layers=4,
    attention_heads=8,
    feedforward_dim=768,
    dropout=0.12,
)
model = All156TemporalNetwork(config)
print({
    "registered_model": "all618_fusion",
    "parameters": sum(parameter.numel() for parameter in model.parameters()),
    "architecture": "bar156 temporal tower + candidate462 masked tower + DeepSets",
})

## Formal unified experiment command

In [ ]:
if not candidate_ready:
    print("BLOCKED: upstream artifact is not candidate462-complete")
else:
    print("bash run_all618_fusion_suite.sh")
    print({
        "routes": ["all618_fusion", "all618_mlp", "all618_lightgbm"],
        "j_policy": "standalone J + mandatory tree delta-J + ensemble J",
        "training_history": "expanding from 2019",
    })